# Variational Information Flow Dynamics for LLM-Guided Tourism Decision Intelligence

This notebook develops a theory-driven simulation framework for tourism decision dynamics. A travel decision is represented as an evolving probability distribution over a decision landscape. The central objective is not only to recommend a final travel option, but to model how the decision trajectory evolves under preference drift, uncertainty diffusion, cost--utility attraction, and LLM-guided constraint correction.

Expected outcomes include: (i) a formal decision-field simulation, (ii) a mathematically defined LLM operator, (iii) trajectory-level evaluation metrics, (iv) sensitivity analysis for robustness, (v) LLM-operator diagnostics, and (vi) a 3D extension to reduce the risk that results are artifacts of a simple 2D geometry.

## 1. Configuration and Colab/Drive Setup

This cell defines the project configuration, mounts Google Drive when available, creates output folders, and initializes reproducibility logs. If Google Drive is not available, the notebook safely falls back to the local Colab runtime.

In [ ]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "VIFD_LLM_Tourism_Decision_Dynamics_v4"
DATASET_PATH = "/content/drive/MyDrive/Datasets/TravelerTrip/TravelDetailsDataset.csv"

np.random.seed(RANDOM_SEED)

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        BASE_DIR = Path("/content/drive/MyDrive/Outputs") / PROJECT_NAME
        print("✅ Google Drive mounted successfully.")
    except Exception as e:
        print("⚠️ Google Drive mount failed:", e)
        print("➡️ Falling back to local runtime under /content.")
        LOCAL_ROOT = Path("/content") if os.access("/content", os.W_OK) else Path("/mnt/data")
        BASE_DIR = LOCAL_ROOT / PROJECT_NAME
else:
    LOCAL_ROOT = Path("/content") if os.access("/content", os.W_OK) else Path("/mnt/data")
    BASE_DIR = LOCAL_ROOT / PROJECT_NAME
    print("ℹ️ Using local runtime because USE_GOOGLE_DRIVE=False.")

FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
OUTPUT_DIR = BASE_DIR / "outputs"
LOG_DIR = BASE_DIR / "logs"

for d in [BASE_DIR, FIG_DIR, TABLE_DIR, OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY_PATH = OUTPUT_DIR / "outputs_summary.txt"
PROGRESS_LOG_PATH = LOG_DIR / "simulation_progress_log.csv"

with open(OUTPUT_SUMMARY_PATH, "w", encoding="utf-8") as f:
    f.write("VIFD-LLM TOURISM DECISION DYNAMICS OUTPUTS SUMMARY\n")
    f.write("=" * 72 + "\n\n")
    f.write("[PROJECT CONFIGURATION]\n")
    f.write(f"Project name: {PROJECT_NAME}\n")
    f.write(f"Random seed: {RANDOM_SEED}\n")
    f.write(f"Use Google Drive: {USE_GOOGLE_DRIVE}\n")
    f.write(f"Base directory: {BASE_DIR}\n")
    f.write(f"Dataset path: {DATASET_PATH}\n\n")

print("📁 Output folders created")
print("BASE_DIR:", BASE_DIR)
print("FIG_DIR:", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("LOG_DIR:", LOG_DIR)
print("OUTPUT_SUMMARY_PATH:", OUTPUT_SUMMARY_PATH)

## 2. Dataset Loading and Validation

The dataset is optional. If available, it can be used later to calibrate or weakly anchor the decision landscape. If unavailable, the notebook proceeds as a theory-driven simulation, which is appropriate for testing the mathematical framework.

In [ ]:
dataset_used = False
df = None
DATASET_EXISTS = os.path.exists(DATASET_PATH)

if DATASET_EXISTS:
    try:
        df = pd.read_csv(DATASET_PATH)
        dataset_used = True
        print(f"✅ Dataset found and loaded: {DATASET_PATH}")
        print("Shape:", df.shape)
        try:
            from IPython.display import display
            display(df.head())
        except Exception:
            print(df.head())
    except Exception as e:
        dataset_used = False
        print(f"⚠️ Dataset found but could not be read: {DATASET_PATH}")
        print("Reason:", e)
        print("➡️ Proceeding with synthetic decision landscape.")
else:
    print(f"⚠️ Dataset NOT found at: {DATASET_PATH}")
    print("➡️ Proceeding with synthetic decision landscape.")

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[DATASET STATUS]\n")
    f.write(f"Dataset exists: {DATASET_EXISTS}\n")
    f.write(f"Dataset used: {dataset_used}\n")
    f.write(f"Dataset path: {DATASET_PATH}\n")
    f.write("Interpretation: the theoretical model is not dependent on dataset availability.\n\n")

## 3. Parameter Grounding and Observable Proxies

This cell explicitly maps the mathematical terms of the model to observable quantities. This addresses the model-validity concern: the framework is not presented as arbitrary mathematics, but as a structure whose parameters can be anchored to behavioral, transactional, or session-level evidence.

In [ ]:
proxy_rows = [
    {
        "Model component": "Drift field v(x,t)",
        "Meaning": "Systematic preference movement",
        "Observable proxy": "Changes in user clicks, repeated visits, time-on-option, booking trajectory, trend shift",
        "Notebook implementation": "Designed vector field with sensitivity-tested magnitude"
    },
    {
        "Model component": "Diffusion coefficient/tensor D",
        "Meaning": "Decision uncertainty and inconsistency",
        "Observable proxy": "Choice variance, repeated-choice inconsistency, browsing dispersion, entropy of candidate set",
        "Notebook implementation": "Scalar D in 2D; sensitivity-tested across regimes"
    },
    {
        "Model component": "Functional J(x)",
        "Meaning": "Cost--regret--uncertainty--constraint landscape",
        "Observable proxy": "Trip cost, mismatch between stated and selected preferences, infeasibility flags",
        "Notebook implementation": "Constructed prior with interpretable components C, R, U, P"
    },
    {
        "Model component": "LLM operator L",
        "Meaning": "Constraint interpretation and policy shaping",
        "Observable proxy": "Feasibility correction, entropy reduction, contraction, idempotency",
        "Notebook implementation": "Composite operator L = P_C o S_lambda"
    }
]

proxy_df = pd.DataFrame(proxy_rows)
proxy_df.to_csv(TABLE_DIR / "parameter_grounding_observable_proxies.csv", index=False)

try:
    from IPython.display import display
    display(proxy_df)
except Exception:
    print(proxy_df)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[PARAMETER GROUNDING]\n")
    f.write("The model components are mapped to observable proxies to reduce the risk of purely decorative mathematical abstraction.\n")
    f.write("Saved table: parameter_grounding_observable_proxies.csv\n\n")

## 4. Define the 2D Decision Landscape

The 2D landscape is used for interpretability and visualization. The two axes represent normalized cost pressure and flexibility/comfort preference. The functional \(J(x)\) combines cost, regret/mismatch, uncertainty, and constraint penalties.

In [ ]:
N = 70
x = np.linspace(0.0, 1.0, N)
y = np.linspace(0.0, 1.0, N)
X, Y = np.meshgrid(x, y, indexing="ij")
dx = x[1] - x[0]
dy = y[1] - y[0]

C = X
R = (Y - 0.72) ** 2 + 0.35 * (X - 0.35) ** 2
U = np.exp(-((X - 0.55) ** 2 + (Y - 0.48) ** 2) / 0.035)
Penalty = np.maximum(0.0, X - 0.70) ** 2 + 0.70 * np.maximum(0.0, 0.25 - Y) ** 2

weights = {
    "alpha_cost": 0.42,
    "beta_regret": 0.27,
    "gamma_uncertainty": 0.16,
    "lambda_penalty": 1.15
}

J_raw = (
    weights["alpha_cost"] * C
    + weights["beta_regret"] * R
    + weights["gamma_uncertainty"] * U
    + weights["lambda_penalty"] * Penalty
)
J = (J_raw - J_raw.min()) / (J_raw.max() - J_raw.min() + 1e-12)

constraint_mask = (X <= 0.70) & (Y >= 0.25)
infeasible_mask = ~constraint_mask
dJdx, dJdy = np.gradient(J, dx, dy, edge_order=2)

sigma0 = 0.13
pi0 = np.exp(-((X - 0.62) ** 2 + (Y - 0.47) ** 2) / (2 * sigma0 ** 2))
pi0 = pi0 / pi0.sum()

weights_df = pd.DataFrame([weights])
weights_df.to_csv(TABLE_DIR / "functional_weights.csv", index=False)

plt.figure(figsize=(7, 5))
plt.contourf(X, Y, J, levels=40)
plt.colorbar(label="Decision functional J(x)")
plt.contour(X, Y, constraint_mask.astype(float), levels=[0.5], linewidths=2)
plt.xlabel("Normalized cost pressure")
plt.ylabel("Flexibility / comfort preference")
plt.title("Figure 1. Tourism decision landscape with feasible boundary")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_1_decision_landscape.png", dpi=300)
plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FIGURE 1 INTERPRETATION]\n")
    f.write("Figure 1 defines the decision landscape J(x). The optimal basin is not merely the cheapest point; it emerges from the balance between cost, preference mismatch, uncertainty, and constraint penalty.\n")
    f.write("The black boundary indicates the feasible decision region used by the LLM projection operator.\n\n")

## 5. Define Metrics and Numerical Operators

This cell defines the trajectory-level metrics used throughout the notebook: entropy, expected objective, optimality gap, regret, total variation stability, infeasible mass, and time-to-epsilon optimality.

In [ ]:
EPS = 1e-12
J_min = float(J.min())

def normalize(P):
    P = np.nan_to_num(P, nan=0.0, posinf=0.0, neginf=0.0)
    P = np.maximum(P, 0.0)
    s = float(P.sum())
    if s <= EPS:
        P = np.ones_like(P) / P.size
    else:
        P = P / s
    return P

def entropy(P):
    P = normalize(P)
    return float(-np.sum(P * np.log(P + EPS)))

def expected_J(P):
    P = normalize(P)
    return float(np.sum(P * J))

def optimality_gap(P):
    return float(expected_J(P) - J_min)

def regret(P):
    P = normalize(P)
    i_star, j_star = np.unravel_index(np.argmin(J), J.shape)
    x_star, y_star = X[i_star, j_star], Y[i_star, j_star]
    dist2 = (X - x_star) ** 2 + (Y - y_star) ** 2
    return float(np.sum(P * dist2))

def total_variation(P, Q):
    P = normalize(P)
    Q = normalize(Q)
    return float(0.5 * np.sum(np.abs(P - Q)))

def infeasible_mass(P):
    P = normalize(P)
    return float(P[infeasible_mask].sum())

def expectation_xy(P):
    P = normalize(P)
    return float(np.sum(P * X)), float(np.sum(P * Y))

def laplacian_2d(P):
    return (
        (np.roll(P, -1, axis=0) - 2 * P + np.roll(P, 1, axis=0)) / (dx ** 2)
        + (np.roll(P, -1, axis=1) - 2 * P + np.roll(P, 1, axis=1)) / (dy ** 2)
    )

def divergence_2d(Ax, Ay):
    dAx_dx = (np.roll(Ax, -1, axis=0) - np.roll(Ax, 1, axis=0)) / (2 * dx)
    dAy_dy = (np.roll(Ay, -1, axis=1) - np.roll(Ay, 1, axis=1)) / (2 * dy)
    return dAx_dx + dAy_dy

def time_to_epsilon_gap(metric_series, epsilon=0.02):
    arr = np.asarray(metric_series)
    idx = np.where(arr <= epsilon)[0]
    if len(idx) == 0:
        return np.nan
    return int(idx[0])

print("Metrics and numerical operators defined.")

## 6. Formal LLM Operator

The LLM layer is formalized as a composite operator \(L = P_C \circ S_\lambda\). The soft shaping operator \(S_\lambda\) reduces mass around high-objective/high-penalty states, while \(P_C\) projects probability mass toward the feasible constraint set. This avoids treating the LLM as an ad-hoc chatbot and makes it measurable.

In [ ]:
Penalty_normalized = (Penalty - Penalty.min()) / (Penalty.max() - Penalty.min() + EPS)

def soft_shaping_operator(P, lambda_shape=2.0, penalty_boost=2.0):
    Q = P * np.exp(-lambda_shape * (J + penalty_boost * Penalty_normalized))
    return normalize(Q)

def projection_operator(P, projection_strength=0.88):
    Q = P.copy()
    Q[infeasible_mask] *= (1.0 - projection_strength)
    return normalize(Q)

def llm_policy_operator(P, lambda_shape=1.7, projection_strength=0.85, penalty_boost=2.0):
    Q = soft_shaping_operator(P, lambda_shape=lambda_shape, penalty_boost=penalty_boost)
    Q = projection_operator(Q, projection_strength=projection_strength)
    return normalize(Q)

def llm_operator_diagnostics(P, lambda_shape=1.7, projection_strength=0.85):
    Q = llm_policy_operator(P, lambda_shape=lambda_shape, projection_strength=projection_strength)
    QQ = llm_policy_operator(Q, lambda_shape=lambda_shape, projection_strength=projection_strength)
    return {
        "entropy_before": entropy(P),
        "entropy_after": entropy(Q),
        "entropy_change": entropy(Q) - entropy(P),
        "infeasible_mass_before": infeasible_mass(P),
        "infeasible_mass_after": infeasible_mass(Q),
        "infeasible_mass_change": infeasible_mass(Q) - infeasible_mass(P),
        "idempotency_tv": total_variation(Q, QQ),
        "expected_J_before": expected_J(P),
        "expected_J_after": expected_J(Q),
        "expected_J_change": expected_J(Q) - expected_J(P)
    }

diagnostic_preview = pd.DataFrame([llm_operator_diagnostics(pi0)])
diagnostic_preview.to_csv(TABLE_DIR / "llm_operator_initial_diagnostics.csv", index=False)

try:
    from IPython.display import display
    display(diagnostic_preview)
except Exception:
    print(diagnostic_preview)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FORMAL LLM OPERATOR]\n")
    f.write("The LLM layer is defined as L = P_C o S_lambda, combining soft policy shaping with feasibility projection.\n")
    f.write("This allows the operator to be evaluated through entropy change, infeasible-mass reduction, idempotency, and objective improvement.\n\n")

## 7. Dynamic Decision Models

This cell defines four systems: an oracle-like static recommender, a greedy ML-only model, a variational flow model, and the LLM-guided variational flow model. The static recommender is retained only as an upper-bound baseline, not as a realistic decision-process model.

In [ ]:
def static_recommender_distribution(width=0.035):
    i_star, j_star = np.unravel_index(np.argmin(J), J.shape)
    xs, ys = X[i_star, j_star], Y[i_star, j_star]
    Q = np.exp(-((X - xs) ** 2 + (Y - ys) ** 2) / (2 * width ** 2))
    return normalize(Q)

def step_greedy_ml(P, eta=2.3):
    Q = P * np.exp(-eta * J)
    return normalize(Q)

def preference_drift_field(drift_scale=0.18):
    Vx = -drift_scale * (0.55 + 0.45 * Y)
    Vy = drift_scale * (0.30 + 0.70 * (1.0 - X))
    return Vx, Vy

def step_variational(P, diffusion=0.00042, kappa=0.022, drift_scale=0.18, dt=0.006, use_llm=False, lambda_shape=1.7, projection_strength=0.85):
    Vx, Vy = preference_drift_field(drift_scale=drift_scale)
    drift_term = -divergence_2d(Vx * P, Vy * P)
    diffusion_term = diffusion * laplacian_2d(P)
    attraction_term = kappa * divergence_2d(P * dJdx, P * dJdy)
    Q = P + dt * (drift_term + diffusion_term + attraction_term)
    Q = normalize(Q)

    if use_llm:
        Q = llm_policy_operator(Q, lambda_shape=lambda_shape, projection_strength=projection_strength)

    return normalize(Q)

print("Dynamic models defined.")

## 8. Run the Main Simulation With Progress Logging

The simulation tracks the evolution of each decision model over time. Progress messages are printed to prevent silent long-running execution, and a CSV log is saved for reproducibility.

In [ ]:
T = 220
PRINT_EVERY = 20

models = {
    "Static recommender": [],
    "Greedy ML-only": [],
    "Variational flow": [],
    "LLM-guided variational flow": []
}

progress_rows = []

def log_progress(step, name, P, prev_P):
    ex, ey = expectation_xy(P)
    row = {
        "step": step,
        "model": name,
        "entropy": entropy(P),
        "expected_J": expected_J(P),
        "optimality_gap": optimality_gap(P),
        "regret": regret(P),
        "stability_tv": total_variation(P, prev_P) if prev_P is not None else 0.0,
        "infeasible_mass": infeasible_mass(P),
        "expected_x_cost": ex,
        "expected_y_flexibility": ey,
        "probability_mass": float(P.sum()),
        "min_probability": float(P.min()),
        "max_probability": float(P.max())
    }
    progress_rows.append(row)
    return row

P_static = static_recommender_distribution()
models["Static recommender"] = [P_static.copy() for _ in range(T + 1)]

P_greedy = pi0.copy()
P_var = pi0.copy()
P_llm = pi0.copy()

models["Greedy ML-only"].append(P_greedy.copy())
models["Variational flow"].append(P_var.copy())
models["LLM-guided variational flow"].append(P_llm.copy())

log_progress(0, "Static recommender", P_static, None)
log_progress(0, "Greedy ML-only", P_greedy, None)
log_progress(0, "Variational flow", P_var, None)
log_progress(0, "LLM-guided variational flow", P_llm, None)

print("Starting dynamic decision simulations...")
print(f"Total steps: {T}; progress printed every {PRINT_EVERY} steps.\n")

start_time = time.time()

for t in range(1, T + 1):
    prev_greedy = P_greedy.copy()
    prev_var = P_var.copy()
    prev_llm = P_llm.copy()

    P_greedy = step_greedy_ml(P_greedy)
    P_var = step_variational(P_var, use_llm=False)
    P_llm = step_variational(P_llm, use_llm=True)

    models["Greedy ML-only"].append(P_greedy.copy())
    models["Variational flow"].append(P_var.copy())
    models["LLM-guided variational flow"].append(P_llm.copy())

    if (t % PRINT_EVERY == 0) or (t == 1) or (t == T):
        rows = [
            log_progress(t, "Static recommender", P_static, P_static),
            log_progress(t, "Greedy ML-only", P_greedy, prev_greedy),
            log_progress(t, "Variational flow", P_var, prev_var),
            log_progress(t, "LLM-guided variational flow", P_llm, prev_llm),
        ]

        elapsed = time.time() - start_time
        print(f"Step {t:03d}/{T} | elapsed={elapsed:.1f}s")
        for r in rows:
            print(
                f"  {r['model']:<30s} "
                f"H={r['entropy']:.4f} | "
                f"J={r['expected_J']:.4f} | "
                f"gap={r['optimality_gap']:.4f} | "
                f"regret={r['regret']:.4f} | "
                f"TV={r['stability_tv']:.6f} | "
                f"infeasible={r['infeasible_mass']:.4f} | "
                f"mass={r['probability_mass']:.6f}"
            )

progress_df = pd.DataFrame(progress_rows)
progress_df.to_csv(PROGRESS_LOG_PATH, index=False)

print("\nAll simulations completed.")
print("Progress log saved to:", PROGRESS_LOG_PATH)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[SIMULATION STATUS]\n")
    f.write(f"Total simulation steps: {T}\n")
    f.write(f"Progress printed every: {PRINT_EVERY} steps\n")
    f.write(f"Progress log saved to: {PROGRESS_LOG_PATH}\n")
    f.write("Simulation completed successfully.\n\n")

## 9. Build Full Trajectory Metrics

This cell computes metrics for every time step and every model. It enables trajectory-level evaluation rather than judging models only by the final selected point.

In [ ]:
metric_rows = []

for model_name, traj in models.items():
    prev = None
    for t, Pdist in enumerate(traj):
        ex, ey = expectation_xy(Pdist)
        metric_rows.append({
            "step": t,
            "model": model_name,
            "entropy": entropy(Pdist),
            "expected_J": expected_J(Pdist),
            "optimality_gap": optimality_gap(Pdist),
            "regret": regret(Pdist),
            "stability_tv": total_variation(Pdist, prev) if prev is not None else 0.0,
            "infeasible_mass": infeasible_mass(Pdist),
            "expected_x_cost": ex,
            "expected_y_flexibility": ey
        })
        prev = Pdist.copy()

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(TABLE_DIR / "dynamic_metrics.csv", index=False)

summary_rows = []
for model_name in models.keys():
    sub = metrics_df[metrics_df["model"] == model_name].reset_index(drop=True)
    summary_rows.append({
        "model": model_name,
        "final_expected_J": float(sub["expected_J"].iloc[-1]),
        "final_optimality_gap": float(sub["optimality_gap"].iloc[-1]),
        "final_regret": float(sub["regret"].iloc[-1]),
        "final_entropy": float(sub["entropy"].iloc[-1]),
        "entropy_reduction": float(sub["entropy"].iloc[0] - sub["entropy"].iloc[-1]),
        "mean_stability_tv": float(sub["stability_tv"].iloc[1:].mean()),
        "final_infeasible_mass": float(sub["infeasible_mass"].iloc[-1]),
        "time_to_gap_0_02": time_to_epsilon_gap(sub["optimality_gap"].values, epsilon=0.02),
        "time_to_gap_0_05": time_to_epsilon_gap(sub["optimality_gap"].values, epsilon=0.05)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLE_DIR / "final_model_summary.csv", index=False)

try:
    from IPython.display import display
    display(summary_df)
except Exception:
    print(summary_df)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[TRAJECTORY METRICS]\n")
    f.write("Full trajectory metrics saved to dynamic_metrics.csv.\n")
    f.write("Final comparative summary saved to final_model_summary.csv.\n")
    f.write("Metrics include entropy, expected objective, optimality gap, regret, stability, infeasible mass, and time-to-epsilon optimality.\n\n")

## 10. Visualize Entropy, Optimality Gap, Regret, Infeasibility, and Stability

These figures provide the main evidence that decision intelligence should be evaluated dynamically. The goal is not merely to reach a good final state, but to move through uncertainty in a stable, interpretable, and constraint-compatible manner.

In [ ]:
plot_specs = [
    ("entropy", "Figure 2. Entropy reduction over decision evolution", "Entropy H(pi)", "figure_2_entropy_reduction.png"),
    ("optimality_gap", "Figure 3. Optimality gap over decision evolution", "Optimality gap", "figure_3_optimality_gap.png"),
    ("regret", "Figure 4. Regret minimization over decision evolution", "Regret", "figure_4_regret_minimization.png"),
    ("infeasible_mass", "Figure 5. Infeasible probability mass over time", "Infeasible mass", "figure_5_infeasible_mass.png"),
    ("stability_tv", "Figure 6. Trajectory stability by total variation", "Total variation", "figure_6_stability_oscillation.png"),
]

for metric, title, ylabel, filename in plot_specs:
    plt.figure(figsize=(7, 4.5))
    for model_name in models.keys():
        sub = metrics_df[metrics_df["model"] == model_name]
        plt.plot(sub["step"], sub[metric], label=model_name)
    plt.xlabel("Decision evolution step")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300)
    plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FIGURES 2-6 INTERPRETATION]\n")
    f.write("Figures 2-6 compare dynamic decision behavior. Entropy reduction reflects clarification of the decision process; optimality gap and regret measure movement toward better basins; infeasible mass measures constraint compatibility; total variation measures oscillation and trajectory stability.\n")
    f.write("The static recommender should be interpreted as an oracle-like upper bound, not as a true decision-process model.\n\n")

## 11. Decision Trajectories in the Landscape

This figure overlays expected decision positions on the landscape. It visually distinguishes a one-shot oracle from models that describe how a traveler moves through the decision space.

In [ ]:
plt.figure(figsize=(7, 5))
plt.contourf(X, Y, J, levels=40, alpha=0.85)
plt.colorbar(label="Decision functional J(x)")
plt.contour(X, Y, constraint_mask.astype(float), levels=[0.5], linewidths=2)

for model_name in models.keys():
    sub = metrics_df[metrics_df["model"] == model_name]
    plt.plot(sub["expected_x_cost"], sub["expected_y_flexibility"], marker="o", markevery=25, label=model_name)

plt.xlabel("Expected normalized cost pressure")
plt.ylabel("Expected flexibility / comfort preference")
plt.title("Figure 7. Expected decision trajectories over the landscape")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_7_decision_trajectories.png", dpi=300)
plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FIGURE 7 INTERPRETATION]\n")
    f.write("Figure 7 shows tourism planning as a trajectory rather than a one-shot recommendation. The proposed models describe how decisions move across the landscape under uncertainty, preference drift, variational attraction, and constraint correction.\n\n")

## 12. Probability Snapshots of the LLM-Guided Variational Flow

This section saves probability snapshots at selected steps to show how probability mass is progressively reallocated from uncertain or infeasible regions toward feasible decision basins.

In [ ]:
snapshot_steps = [0, 30, 80, 140, 220]
for step in snapshot_steps:
    P_snap = models["LLM-guided variational flow"][step]
    plt.figure(figsize=(6, 5))
    plt.contourf(X, Y, P_snap, levels=40)
    plt.colorbar(label="Probability mass")
    plt.contour(X, Y, constraint_mask.astype(float), levels=[0.5], linewidths=2)
    plt.xlabel("Normalized cost pressure")
    plt.ylabel("Flexibility / comfort preference")
    plt.title(f"Figure 8. LLM-guided probability field at step {step}")
    plt.tight_layout()
    fname = f"figure_8_probability_snapshot_step_{step}.png"
    plt.savefig(FIG_DIR / fname, dpi=300)
    plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FIGURE 8 INTERPRETATION]\n")
    f.write("Figure 8 illustrates dynamic decision concentration. The LLM-guided operator reallocates probability mass toward feasible, preference-compatible regions while the variational flow controls movement over the objective landscape.\n\n")

## 13. LLM Operator Diagnostics

This cell evaluates the formal LLM operator directly. It reports entropy change, infeasible-mass reduction, objective change, contraction behavior, and idempotency. These diagnostics make the LLM component mathematically inspectable.

In [ ]:
def random_distribution_on_grid(smooth_steps=3):
    Q = np.random.rand(N, N)
    Q = normalize(Q)
    for _ in range(smooth_steps):
        Q = normalize(Q + 0.08 * laplacian_2d(Q))
    return normalize(Q)

diagnostic_rows = []

for sample_id in range(30):
    A = random_distribution_on_grid()
    B = random_distribution_on_grid()

    LA = llm_policy_operator(A)
    LB = llm_policy_operator(B)

    diag = llm_operator_diagnostics(A)
    dist_before = total_variation(A, B)
    dist_after = total_variation(LA, LB)
    contraction_ratio = dist_after / (dist_before + EPS)

    diag.update({
        "sample_id": sample_id,
        "contraction_ratio": contraction_ratio,
        "dist_before": dist_before,
        "dist_after": dist_after
    })
    diagnostic_rows.append(diag)

llm_diag_df = pd.DataFrame(diagnostic_rows)
llm_diag_df.to_csv(TABLE_DIR / "llm_operator_diagnostics.csv", index=False)

llm_diag_summary = llm_diag_df.mean(numeric_only=True).to_frame("mean_value")
llm_diag_summary.to_csv(TABLE_DIR / "llm_operator_diagnostics_summary.csv")

try:
    from IPython.display import display
    display(llm_diag_summary)
except Exception:
    print(llm_diag_summary)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[LLM OPERATOR DIAGNOSTICS]\n")
    f.write("LLM operator diagnostics saved to llm_operator_diagnostics.csv and llm_operator_diagnostics_summary.csv.\n")
    f.write("The operator is evaluated through entropy change, infeasible-mass reduction, objective change, contraction ratio, and idempotency.\n")
    f.write(llm_diag_summary.to_string())
    f.write("\n\n")

## 14. Sensitivity Analysis

This section tests whether the qualitative findings persist across plausible parameter regimes. This directly addresses the concern that the framework may only work because of a single hand-designed parameter setting.

In [ ]:
def run_short_simulation(diffusion=0.00042, drift_scale=0.18, lambda_shape=1.7, projection_strength=0.85, steps=90):
    Pdist = pi0.copy()
    stability_values = []

    for _ in range(steps):
        old = Pdist.copy()
        Pdist = step_variational(
            Pdist,
            diffusion=diffusion,
            drift_scale=drift_scale,
            use_llm=True,
            lambda_shape=lambda_shape,
            projection_strength=projection_strength
        )
        stability_values.append(total_variation(Pdist, old))

    return {
        "final_expected_J": expected_J(Pdist),
        "final_optimality_gap": optimality_gap(Pdist),
        "final_regret": regret(Pdist),
        "final_entropy": entropy(Pdist),
        "final_infeasible_mass": infeasible_mass(Pdist),
        "mean_stability_tv": float(np.mean(stability_values)),
        "max_stability_tv": float(np.max(stability_values))
    }

sensitivity_rows = []
diffusion_grid = [0.00018, 0.00042, 0.00080]
drift_grid = [0.10, 0.18, 0.28]
lambda_grid = [0.9, 1.7, 2.6]
projection_grid = [0.65, 0.85]

total_runs = len(diffusion_grid) * len(drift_grid) * len(lambda_grid) * len(projection_grid)
run_id = 0

print(f"Starting sensitivity analysis with {total_runs} runs.")

for D_val in diffusion_grid:
    for drift_val in drift_grid:
        for lambda_val in lambda_grid:
            for proj_val in projection_grid:
                run_id += 1
                result = run_short_simulation(
                    diffusion=D_val,
                    drift_scale=drift_val,
                    lambda_shape=lambda_val,
                    projection_strength=proj_val,
                    steps=90
                )
                result.update({
                    "run_id": run_id,
                    "diffusion": D_val,
                    "drift_scale": drift_val,
                    "lambda_shape": lambda_val,
                    "projection_strength": proj_val
                })
                sensitivity_rows.append(result)

                if run_id % 6 == 0 or run_id == total_runs:
                    print(f"Completed sensitivity run {run_id}/{total_runs}")

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(TABLE_DIR / "sensitivity_analysis.csv", index=False)

robustness_summary = sensitivity_df[
    [
        "final_expected_J",
        "final_optimality_gap",
        "final_regret",
        "final_entropy",
        "final_infeasible_mass",
        "mean_stability_tv"
    ]
].describe().T

robustness_summary.to_csv(TABLE_DIR / "sensitivity_analysis_summary.csv")

try:
    from IPython.display import display
    display(robustness_summary)
except Exception:
    print(robustness_summary)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[SENSITIVITY ANALYSIS]\n")
    f.write("Sensitivity analysis varied diffusion, drift strength, LLM shaping strength, and feasibility projection strength.\n")
    f.write("This tests whether qualitative behavior persists across parameter regimes rather than only under one designed configuration.\n")
    f.write("Saved files: sensitivity_analysis.csv and sensitivity_analysis_summary.csv.\n\n")

## 15. Sensitivity Analysis Visuals

These figures show the distribution of outcomes across parameter regimes. The purpose is to demonstrate robustness and identify fragile regions of the model.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(sensitivity_df["final_optimality_gap"], sensitivity_df["final_infeasible_mass"])
plt.xlabel("Final optimality gap")
plt.ylabel("Final infeasible mass")
plt.title("Figure 9. Robustness trade-off across sensitivity regimes")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_9_sensitivity_tradeoff.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 4.5))
plt.scatter(sensitivity_df["mean_stability_tv"], sensitivity_df["final_entropy"])
plt.xlabel("Mean trajectory variation")
plt.ylabel("Final entropy")
plt.title("Figure 10. Stability--uncertainty behavior across regimes")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_10_sensitivity_stability_entropy.png", dpi=300)
plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FIGURES 9-10 INTERPRETATION]\n")
    f.write("Figures 9-10 summarize the robustness analysis. The key scientific question is whether low infeasibility, reduced uncertainty, and stable trajectories persist under parameter perturbations.\n\n")

## 16. 3D Decision-Field Extension

The 3D extension reduces the risk that the results are artifacts of a simple 2D geometry. The third dimension represents timing uncertainty or schedule compatibility.

In [ ]:
N3 = 22
x3 = np.linspace(0.0, 1.0, N3)
y3 = np.linspace(0.0, 1.0, N3)
z3 = np.linspace(0.0, 1.0, N3)
X3, Y3, Z3 = np.meshgrid(x3, y3, z3, indexing="ij")
h3 = x3[1] - x3[0]

C3 = X3
R3 = (Y3 - 0.70) ** 2 + 0.25 * (X3 - 0.35) ** 2 + 0.35 * (Z3 - 0.55) ** 2
U3 = np.exp(-((X3 - 0.55) ** 2 + (Y3 - 0.48) ** 2 + (Z3 - 0.50) ** 2) / 0.055)
Penalty3 = np.maximum(0.0, X3 - 0.70) ** 2 + 0.65 * np.maximum(0.0, 0.25 - Y3) ** 2 + 0.35 * np.maximum(0.0, np.abs(Z3 - 0.55) - 0.35) ** 2

J3_raw = 0.42 * C3 + 0.27 * R3 + 0.16 * U3 + 1.15 * Penalty3
J3 = (J3_raw - J3_raw.min()) / (J3_raw.max() - J3_raw.min() + EPS)
constraint3 = (X3 <= 0.70) & (Y3 >= 0.25) & (np.abs(Z3 - 0.55) <= 0.35)
infeasible3 = ~constraint3

dJ3dx, dJ3dy, dJ3dz = np.gradient(J3, h3, h3, h3, edge_order=2)

P3_init = np.exp(-((X3 - 0.62) ** 2 + (Y3 - 0.47) ** 2 + (Z3 - 0.42) ** 2) / (2 * 0.16 ** 2))
P3_init = P3_init / P3_init.sum()

def normalize3(Pdist):
    Pdist = np.nan_to_num(Pdist, nan=0.0, posinf=0.0, neginf=0.0)
    Pdist = np.maximum(Pdist, 0.0)
    s = float(Pdist.sum())
    if s <= EPS:
        return np.ones_like(Pdist) / Pdist.size
    return Pdist / s

def entropy3(Pdist):
    Pdist = normalize3(Pdist)
    return float(-np.sum(Pdist * np.log(Pdist + EPS)))

def expected_J3(Pdist):
    Pdist = normalize3(Pdist)
    return float(np.sum(Pdist * J3))

def infeasible_mass3(Pdist):
    Pdist = normalize3(Pdist)
    return float(Pdist[infeasible3].sum())

def laplacian3(Pdist):
    return (
        (np.roll(Pdist, -1, axis=0) - 2 * Pdist + np.roll(Pdist, 1, axis=0)) / (h3 ** 2)
        + (np.roll(Pdist, -1, axis=1) - 2 * Pdist + np.roll(Pdist, 1, axis=1)) / (h3 ** 2)
        + (np.roll(Pdist, -1, axis=2) - 2 * Pdist + np.roll(Pdist, 1, axis=2)) / (h3 ** 2)
    )

def divergence3(Ax, Ay, Az):
    return (
        (np.roll(Ax, -1, axis=0) - np.roll(Ax, 1, axis=0)) / (2 * h3)
        + (np.roll(Ay, -1, axis=1) - np.roll(Ay, 1, axis=1)) / (2 * h3)
        + (np.roll(Az, -1, axis=2) - np.roll(Az, 1, axis=2)) / (2 * h3)
    )

def llm_operator3(Pdist, lambda_shape=1.5, projection_strength=0.82):
    penalty_norm = (Penalty3 - Penalty3.min()) / (Penalty3.max() - Penalty3.min() + EPS)
    Q = Pdist * np.exp(-lambda_shape * (J3 + 2.0 * penalty_norm))
    Q[infeasible3] *= (1.0 - projection_strength)
    return normalize3(Q)

def step3(Pdist, diffusion=0.00030, kappa=0.016, drift_scale=0.12, dt=0.0035, use_llm=True):
    Vx = -drift_scale * (0.45 + 0.35 * Y3)
    Vy = drift_scale * (0.25 + 0.50 * (1.0 - X3))
    Vz = drift_scale * (0.55 - Z3)

    drift = -divergence3(Vx * Pdist, Vy * Pdist, Vz * Pdist)
    diff = diffusion * laplacian3(Pdist)
    attract = kappa * divergence3(Pdist * dJ3dx, Pdist * dJ3dy, Pdist * dJ3dz)

    Q = normalize3(Pdist + dt * (drift + diff + attract))
    if use_llm:
        Q = llm_operator3(Q)
    return normalize3(Q)

T3 = 120
P3_current = P3_init.copy()
metrics3 = []

print("Starting 3D extension simulation.")
for t in range(T3 + 1):
    metrics3.append({
        "step": t,
        "entropy": entropy3(P3_current),
        "expected_J": expected_J3(P3_current),
        "optimality_gap": expected_J3(P3_current) - float(J3.min()),
        "infeasible_mass": infeasible_mass3(P3_current)
    })
    if t < T3:
        P3_current = step3(P3_current, use_llm=True)
    if t % 30 == 0:
        print(f"3D step {t:03d}/{T3}")

metrics3_df = pd.DataFrame(metrics3)
metrics3_df.to_csv(TABLE_DIR / "three_dimensional_extension_metrics.csv", index=False)

plt.figure(figsize=(7, 4.5))
plt.plot(metrics3_df["step"], metrics3_df["entropy"], label="3D entropy")
plt.plot(metrics3_df["step"], metrics3_df["optimality_gap"], label="3D optimality gap")
plt.plot(metrics3_df["step"], metrics3_df["infeasible_mass"], label="3D infeasible mass")
plt.xlabel("Decision evolution step")
plt.ylabel("Metric value")
plt.title("Figure 11. 3D decision-field extension")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_11_3d_extension_metrics.png", dpi=300)
plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[3D EXTENSION]\n")
    f.write("A 3D decision-field extension was added with cost, flexibility, and timing compatibility axes.\n")
    f.write("This reduces the risk that observed behavior is only a 2D artifact.\n")
    f.write("Saved file: three_dimensional_extension_metrics.csv and figure_11_3d_extension_metrics.png.\n\n")

## 17. Final Interpretation and Reproducibility Checklist

This cell writes a manuscript-oriented summary to the outputs file. It emphasizes the correct interpretation: the static recommender is an oracle-like upper bound, while the proposed framework models dynamic decision formation.

In [ ]:
best_final_objective = summary_df.sort_values("final_expected_J").iloc[0]["model"]
best_entropy_reduction = summary_df.sort_values("entropy_reduction", ascending=False).iloc[0]["model"]
best_low_infeasible = summary_df.sort_values("final_infeasible_mass").iloc[0]["model"]
best_stability = summary_df.sort_values("mean_stability_tv").iloc[0]["model"]

reproducibility = {
    "figures": sorted([p.name for p in FIG_DIR.glob("*.png")]),
    "tables": sorted([p.name for p in TABLE_DIR.glob("*.csv")]),
    "logs": sorted([p.name for p in LOG_DIR.glob("*.csv")]),
    "outputs": sorted([p.name for p in OUTPUT_DIR.glob("*.txt")])
}

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[FINAL INTERPRETATION]\n\n")
    f.write(f"Best final objective ranking: {best_final_objective}\n")
    f.write(f"Strongest entropy reduction ranking: {best_entropy_reduction}\n")
    f.write(f"Lowest final infeasible mass ranking: {best_low_infeasible}\n")
    f.write(f"Best mean trajectory stability ranking: {best_stability}\n\n")

    f.write("The static recommender is an oracle-like upper-bound baseline because it directly places mass near the global optimum. It is not a realistic decision-process model and should not be interpreted as evidence against the dynamic framework.\n\n")

    f.write("The scientific objective of this notebook is trajectory-level decision intelligence. The variational flow model represents how probability mass evolves under preference drift, uncertainty diffusion, and gradient-based attraction toward lower values of J(x). The LLM-guided operator is formalized as L = P_C o S_lambda, which combines soft policy shaping with feasibility projection.\n\n")

    f.write("The notebook directly addresses three fundamental limitations. First, model validity is strengthened through an observable-proxy table and parameter sensitivity analysis. Second, the LLM operator is formalized and evaluated through entropy, infeasible-mass reduction, contraction, and idempotency diagnostics. Third, the dimensionality concern is partially addressed through a 3D decision-field extension.\n\n")

    f.write("The results should not be overclaimed as final empirical superiority over deployed recommender systems. Their value is to demonstrate that tourism decision intelligence can be formulated, simulated, diagnosed, and evaluated as a dynamic information-flow process.\n\n")

    f.write("[REPRODUCIBILITY CHECKLIST]\n")
    f.write(json.dumps(reproducibility, indent=2))
    f.write("\n")

print("Final outputs summary written to:", OUTPUT_SUMMARY_PATH)
print(json.dumps(reproducibility, indent=2))

## Controlled Dynamics Upgrade: Regime and Phase Analysis
This section turns the simulation into a controlled dynamical-systems experiment. It sweeps drift and diffusion parameters and classifies the resulting behavior into stable, oscillatory, diffusion-dominated, or collapse-like regimes.

In [ ]:
# =========================================================
# CONTROLLED DYNAMICS: REGIME AND PHASE ANALYSIS
# =========================================================
REGIME_STEPS = 100
REGIME_PRINT_EVERY = 10

D_grid = np.linspace(0.00010, 0.00120, 8)
drift_grid_regime = np.linspace(0.06, 0.34, 8)


def classify_regime(final_entropy, final_infeasible, tail_tv, final_gap, max_probability):
    """Classify decision dynamics into interpretable qualitative regimes."""
    if final_entropy < 2.0 or max_probability > 0.35:
        return "LLM-collapse"
    if final_entropy > 7.25 and final_gap > 0.20:
        return "diffusion-dominated"
    if tail_tv > 0.010:
        return "oscillatory"
    if final_infeasible < 0.08 and tail_tv <= 0.010:
        return "stable-convergent"
    return "transitional"


def simulate_regime_point(diffusion, drift_scale, use_llm=True, lambda_shape=1.7, projection_strength=0.85, steps=REGIME_STEPS):
    Pdist = pi0.copy()
    tv_values = []
    entropy_values = []
    infeasible_values = []
    gap_values = []

    for _ in range(steps):
        old = Pdist.copy()
        Pdist = step_variational(
            Pdist,
            diffusion=diffusion,
            drift_scale=drift_scale,
            use_llm=use_llm,
            lambda_shape=lambda_shape,
            projection_strength=projection_strength
        )
        tv_values.append(total_variation(Pdist, old))
        entropy_values.append(entropy(Pdist))
        infeasible_values.append(infeasible_mass(Pdist))
        gap_values.append(optimality_gap(Pdist))

    tail = max(5, steps // 5)
    final_entropy = entropy_values[-1]
    final_infeasible = infeasible_values[-1]
    tail_tv = float(np.mean(tv_values[-tail:]))
    final_gap = gap_values[-1]
    max_prob = float(Pdist.max())
    regime = classify_regime(final_entropy, final_infeasible, tail_tv, final_gap, max_prob)

    return {
        "diffusion": diffusion,
        "drift_scale": drift_scale,
        "use_llm": use_llm,
        "lambda_shape": lambda_shape,
        "projection_strength": projection_strength,
        "final_entropy": final_entropy,
        "final_infeasible_mass": final_infeasible,
        "final_optimality_gap": final_gap,
        "mean_tail_tv": tail_tv,
        "max_probability": max_prob,
        "regime": regime
    }

regime_rows = []
print("Starting regime analysis over diffusion and drift parameters...")
for i, D_val in enumerate(D_grid, start=1):
    for j, drift_val in enumerate(drift_grid_regime, start=1):
        regime_rows.append(simulate_regime_point(D_val, drift_val, use_llm=True))
    if (i % 2 == 0) or (i == len(D_grid)):
        print(f"Completed diffusion row {i}/{len(D_grid)}")

regime_df = pd.DataFrame(regime_rows)
regime_df.to_csv(TABLE_DIR / "regime_phase_analysis.csv", index=False)

regime_summary = regime_df["regime"].value_counts().rename_axis("regime").reset_index(name="count")
regime_summary.to_csv(TABLE_DIR / "regime_phase_summary.csv", index=False)

print("Regime analysis completed.")
display(regime_summary)
print("Saved:", TABLE_DIR / "regime_phase_analysis.csv")

## Regime Heatmaps
The following figures show where the dynamics are stable, infeasible, or oscillatory as diffusion and drift vary. These plots are designed to support stronger claims about controlled behavior rather than only visual convergence.

In [ ]:
# =========================================================
# FIGURES 12-14: REGIME HEATMAPS
# =========================================================
def pivot_metric(df, metric):
    return df.pivot(index="drift_scale", columns="diffusion", values=metric).sort_index(ascending=True)

heatmap_specs = [
    ("final_optimality_gap", "Figure 12: Phase map of final optimality gap", "figure_12_phase_map_optimality_gap.png"),
    ("final_infeasible_mass", "Figure 13: Phase map of final infeasible mass", "figure_13_phase_map_infeasible_mass.png"),
    ("mean_tail_tv", "Figure 14: Phase map of tail trajectory oscillation", "figure_14_phase_map_tail_tv.png"),
]

for metric, title, filename in heatmap_specs:
    mat = pivot_metric(regime_df, metric)
    plt.figure(figsize=(8, 5.5))
    plt.imshow(mat.values, origin="lower", aspect="auto", extent=[D_grid.min(), D_grid.max(), drift_grid_regime.min(), drift_grid_regime.max()])
    plt.colorbar(label=metric)
    plt.xlabel("Diffusion coefficient D")
    plt.ylabel("Drift strength ||v||")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[CONTROLLED DYNAMICS AND REGIME ANALYSIS]\n")
    f.write("A diffusion--drift phase analysis was added to identify stable, oscillatory, diffusion-dominated, and collapse-like regimes.\n")
    f.write("Saved files: regime_phase_analysis.csv and regime_phase_summary.csv.\n")
    f.write("Saved figures: figure_12_phase_map_optimality_gap.png, figure_13_phase_map_infeasible_mass.png, figure_14_phase_map_tail_tv.png.\n\n")

## LLM Projection Strength Sweep
This section studies whether the LLM operator improves feasibility smoothly or causes premature collapse. The aim is to identify a useful range of the projection strength rather than choosing one arbitrary value.

In [ ]:
# =========================================================
# LLM PROJECTION STRENGTH SWEEP
# =========================================================
lambda_sweep = np.linspace(0.0, 3.0, 13)
projection_sweep = np.linspace(0.0, 0.95, 11)
projection_rows = []

print("Starting LLM operator strength sweep...")
for lam in lambda_sweep:
    for proj in projection_sweep:
        out = simulate_regime_point(
            diffusion=0.00042,
            drift_scale=0.18,
            use_llm=True,
            lambda_shape=float(lam),
            projection_strength=float(proj),
            steps=80
        )
        projection_rows.append(out)

projection_sweep_df = pd.DataFrame(projection_rows)
projection_sweep_df.to_csv(TABLE_DIR / "llm_projection_strength_sweep.csv", index=False)

plt.figure(figsize=(8, 5.5))
mat = projection_sweep_df.pivot(index="projection_strength", columns="lambda_shape", values="final_infeasible_mass")
plt.imshow(mat.values, origin="lower", aspect="auto", extent=[lambda_sweep.min(), lambda_sweep.max(), projection_sweep.min(), projection_sweep.max()])
plt.colorbar(label="Final infeasible mass")
plt.xlabel("Soft shaping strength lambda")
plt.ylabel("Projection strength")
plt.title("Figure 15: Feasibility response to LLM operator strength")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_15_llm_strength_feasibility.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5.5))
mat = projection_sweep_df.pivot(index="projection_strength", columns="lambda_shape", values="final_entropy")
plt.imshow(mat.values, origin="lower", aspect="auto", extent=[lambda_sweep.min(), lambda_sweep.max(), projection_sweep.min(), projection_sweep.max()])
plt.colorbar(label="Final entropy")
plt.xlabel("Soft shaping strength lambda")
plt.ylabel("Projection strength")
plt.title("Figure 16: Entropy response to LLM operator strength")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_16_llm_strength_entropy.png", dpi=300, bbox_inches="tight")
plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[LLM STRENGTH SWEEP]\n")
    f.write("The notebook now varies soft shaping and projection strength to identify useful operator regimes and detect premature collapse.\n")
    f.write("Saved file: llm_projection_strength_sweep.csv.\n")
    f.write("Saved figures: figure_15_llm_strength_feasibility.png and figure_16_llm_strength_entropy.png.\n\n")

print("LLM strength sweep completed and saved.")

## Pareto Trade-Off Frontier
This section transforms the multi-metric results into a trade-off geometry. Each parameter configuration is treated as a point in objective--feasibility--entropy--stability space, and non-dominated points are identified as an approximate Pareto frontier.

In [ ]:
# =========================================================
# PARETO FRONTIER ANALYSIS
# =========================================================
def run_pareto_simulation(diffusion, drift_scale, lambda_shape, projection_strength, use_llm, steps=80):
    Pdist = pi0.copy()
    tv_values = []
    for _ in range(steps):
        old = Pdist.copy()
        Pdist = step_variational(
            Pdist,
            diffusion=diffusion,
            drift_scale=drift_scale,
            use_llm=use_llm,
            lambda_shape=lambda_shape,
            projection_strength=projection_strength
        )
        tv_values.append(total_variation(Pdist, old))

    return {
        "diffusion": diffusion,
        "drift_scale": drift_scale,
        "lambda_shape": lambda_shape,
        "projection_strength": projection_strength,
        "use_llm": use_llm,
        "final_expected_J": expected_J(Pdist),
        "final_feasibility": 1.0 - infeasible_mass(Pdist),
        "final_entropy": entropy(Pdist),
        "mean_stability_tv": float(np.mean(tv_values)),
        "final_optimality_gap": optimality_gap(Pdist)
    }


def pareto_mask_minimize(values):
    """Return mask for non-dominated rows assuming all columns are minimized."""
    n = values.shape[0]
    is_efficient = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_efficient[i]:
            continue
        dominated_by_i = np.all(values[i] <= values, axis=1) & np.any(values[i] < values, axis=1)
        is_efficient[dominated_by_i] = False
    return is_efficient

pareto_rows = []
pareto_D = [0.00018, 0.00042, 0.00080, 0.00110]
pareto_drift = [0.08, 0.16, 0.24, 0.32]
pareto_lambda = [0.0, 1.0, 1.7, 2.4]
pareto_proj = [0.0, 0.55, 0.85]

print("Starting Pareto sweep...")
for D_val in pareto_D:
    for drift_val in pareto_drift:
        for lam in pareto_lambda:
            for proj in pareto_proj:
                use_llm_flag = (lam > 0.0) or (proj > 0.0)
                pareto_rows.append(run_pareto_simulation(D_val, drift_val, lam, proj, use_llm=use_llm_flag, steps=80))

pareto_df = pd.DataFrame(pareto_rows)

# Commitment-oriented Pareto objectives: minimize J, infeasibility, entropy, and instability.
# Entropy can also be treated as diversity in another analysis; here it measures decision commitment.
pareto_objectives = np.column_stack([
    pareto_df["final_expected_J"].values,
    1.0 - pareto_df["final_feasibility"].values,
    pareto_df["final_entropy"].values,
    pareto_df["mean_stability_tv"].values,
])
pareto_df["is_pareto"] = pareto_mask_minimize(pareto_objectives)
pareto_df.to_csv(TABLE_DIR / "pareto_tradeoff_analysis.csv", index=False)
pareto_df[pareto_df["is_pareto"]].to_csv(TABLE_DIR / "pareto_frontier_points.csv", index=False)

print("Pareto sweep completed.")
print("Number of configurations:", len(pareto_df))
print("Number of Pareto frontier points:", int(pareto_df["is_pareto"].sum()))

display(pareto_df.sort_values(["is_pareto", "final_feasibility"], ascending=[False, False]).head(10))

In [ ]:
# =========================================================
# FIGURES 17-18: PARETO TRADE-OFF VISUALIZATION
# =========================================================
front = pareto_df[pareto_df["is_pareto"]].copy()
nonfront = pareto_df[~pareto_df["is_pareto"]].copy()

plt.figure(figsize=(8, 6))
plt.scatter(nonfront["final_feasibility"], nonfront["final_entropy"], alpha=0.35, label="Dominated configurations")
plt.scatter(front["final_feasibility"], front["final_entropy"], marker="x", s=80, label="Approximate Pareto frontier")
plt.xlabel("Final feasibility")
plt.ylabel("Final entropy")
plt.title("Figure 17: Pareto trade-off between feasibility and entropy")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_17_pareto_feasibility_entropy.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(nonfront["final_expected_J"], nonfront["mean_stability_tv"], alpha=0.35, label="Dominated configurations")
plt.scatter(front["final_expected_J"], front["mean_stability_tv"], marker="x", s=80, label="Approximate Pareto frontier")
plt.xlabel("Final expected objective J")
plt.ylabel("Mean trajectory variation")
plt.title("Figure 18: Pareto trade-off between objective value and stability")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_18_pareto_objective_stability.png", dpi=300, bbox_inches="tight")
plt.show()

# Marginal feasibility gain per entropy cost for representative matched LLM/no-LLM pairs.
matched_rows = []
for D_val in pareto_D:
    for drift_val in pareto_drift:
        base = run_pareto_simulation(D_val, drift_val, 0.0, 0.0, use_llm=False, steps=80)
        llm = run_pareto_simulation(D_val, drift_val, 1.7, 0.85, use_llm=True, steps=80)
        delta_feasibility = llm["final_feasibility"] - base["final_feasibility"]
        delta_entropy = llm["final_entropy"] - base["final_entropy"]
        matched_rows.append({
            "diffusion": D_val,
            "drift_scale": drift_val,
            "delta_feasibility": delta_feasibility,
            "delta_entropy": delta_entropy,
            "feasibility_gain_per_entropy_cost": delta_feasibility / (abs(delta_entropy) + EPS),
            "base_feasibility": base["final_feasibility"],
            "llm_feasibility": llm["final_feasibility"],
            "base_entropy": base["final_entropy"],
            "llm_entropy": llm["final_entropy"]
        })

marginal_df = pd.DataFrame(matched_rows)
marginal_df.to_csv(TABLE_DIR / "llm_marginal_tradeoff_rates.csv", index=False)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[PARETO TRADE-OFF ANALYSIS]\n")
    f.write("A Pareto frontier analysis was added to interpret objective value, feasibility, entropy, and stability as a multi-objective decision geometry.\n")
    f.write(f"Total configurations: {len(pareto_df)}; Pareto frontier points: {int(pareto_df['is_pareto'].sum())}.\n")
    f.write("Saved files: pareto_tradeoff_analysis.csv, pareto_frontier_points.csv, llm_marginal_tradeoff_rates.csv.\n")
    f.write("Saved figures: figure_17_pareto_feasibility_entropy.png and figure_18_pareto_objective_stability.png.\n\n")

print("Marginal trade-off rates saved.")
display(marginal_df.describe())

## Mechanism Identification: Persistence and Energy-Landscape Tests
This section asks whether the LLM operator is only a local projection or whether it produces a more persistent redistribution of probability mass. A one-shot LLM intervention is applied and then removed, and the subsequent dynamics are monitored.

In [ ]:
# =========================================================
# MECHANISM TEST 1: PERSISTENCE AFTER REMOVING THE LLM OPERATOR
# =========================================================
P_base = pi0.copy()
P_one_shot = llm_policy_operator(P_base, lambda_shape=1.7, projection_strength=0.85)
P_repeated = P_base.copy()

persistence_rows = []
P_a = P_base.copy()
P_b = P_one_shot.copy()
P_c = P_repeated.copy()

for t in range(0, 121):
    if t > 0:
        P_a = step_variational(P_a, use_llm=False)
        P_b = step_variational(P_b, use_llm=False)
        P_c = step_variational(P_c, use_llm=True, lambda_shape=1.7, projection_strength=0.85)

    persistence_rows.extend([
        {"step": t, "scenario": "No LLM", "infeasible_mass": infeasible_mass(P_a), "entropy": entropy(P_a), "expected_J": expected_J(P_a)},
        {"step": t, "scenario": "One-shot LLM then removed", "infeasible_mass": infeasible_mass(P_b), "entropy": entropy(P_b), "expected_J": expected_J(P_b)},
        {"step": t, "scenario": "Repeated LLM", "infeasible_mass": infeasible_mass(P_c), "entropy": entropy(P_c), "expected_J": expected_J(P_c)},
    ])

persistence_df = pd.DataFrame(persistence_rows)
persistence_df.to_csv(TABLE_DIR / "llm_persistence_after_removal.csv", index=False)

plt.figure(figsize=(8, 5.5))
for scenario, group in persistence_df.groupby("scenario"):
    plt.plot(group["step"], group["infeasible_mass"], label=scenario)
plt.xlabel("Decision step")
plt.ylabel("Infeasible probability mass")
plt.title("Figure 19: Persistence of LLM feasibility correction after removal")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_19_llm_persistence_infeasible_mass.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5.5))
for scenario, group in persistence_df.groupby("scenario"):
    plt.plot(group["step"], group["entropy"], label=scenario)
plt.xlabel("Decision step")
plt.ylabel("Entropy")
plt.title("Figure 20: Persistence of LLM-induced decision commitment")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_20_llm_persistence_entropy.png", dpi=300, bbox_inches="tight")
plt.show()

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[LLM MECHANISM: PERSISTENCE TEST]\n")
    f.write("A one-shot LLM intervention was applied and then removed to test whether feasibility correction persists under subsequent dynamics.\n")
    f.write("Slow decay indicates redistribution or basin reshaping; rapid decay indicates local projection only.\n")
    f.write("Saved file: llm_persistence_after_removal.csv.\n")
    f.write("Saved figures: figure_19_llm_persistence_infeasible_mass.png and figure_20_llm_persistence_entropy.png.\n\n")

In [ ]:
# =========================================================
# MECHANISM TEST 2: EFFECTIVE ENERGY LANDSCAPE RECONSTRUCTION
# =========================================================
def effective_energy(Pdist):
    U = -np.log(normalize(Pdist) + EPS)
    # Clip the extreme upper tail for visualization only.
    return np.minimum(U, np.percentile(U, 98))

P_no_llm_final = P_base.copy()
P_llm_final = P_base.copy()
for _ in range(160):
    P_no_llm_final = step_variational(P_no_llm_final, use_llm=False)
    P_llm_final = step_variational(P_llm_final, use_llm=True, lambda_shape=1.7, projection_strength=0.85)

U_no_llm = effective_energy(P_no_llm_final)
U_llm = effective_energy(P_llm_final)
U_shift = U_llm - U_no_llm

np.save(TABLE_DIR / "effective_energy_no_llm.npy", U_no_llm)
np.save(TABLE_DIR / "effective_energy_llm.npy", U_llm)
np.save(TABLE_DIR / "effective_energy_shift.npy", U_shift)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
ims = []
ims.append(axes[0].contourf(X, Y, U_no_llm, levels=30))
axes[0].set_title("No LLM effective energy")
axes[0].set_xlabel("Cost pressure")
axes[0].set_ylabel("Flexibility")
ims.append(axes[1].contourf(X, Y, U_llm, levels=30))
axes[1].set_title("LLM-guided effective energy")
axes[1].set_xlabel("Cost pressure")
axes[1].set_ylabel("Flexibility")
ims.append(axes[2].contourf(X, Y, U_shift, levels=30))
axes[2].set_title("Energy shift induced by LLM")
axes[2].set_xlabel("Cost pressure")
axes[2].set_ylabel("Flexibility")
for ax in axes:
    ax.contour(X, Y, infeasible_mask.astype(float), levels=[0.5], linewidths=1)
fig.colorbar(ims[1], ax=axes[:2], shrink=0.75, label="Effective energy")
fig.colorbar(ims[2], ax=axes[2], shrink=0.75, label="Energy shift")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_21_effective_energy_landscape_reconstruction.png", dpi=300, bbox_inches="tight")
plt.show()

energy_rows = [{
    "scenario": "No LLM",
    "mean_energy_feasible": float(U_no_llm[~infeasible_mask].mean()),
    "mean_energy_infeasible": float(U_no_llm[infeasible_mask].mean()),
    "energy_gap_infeasible_minus_feasible": float(U_no_llm[infeasible_mask].mean() - U_no_llm[~infeasible_mask].mean())
}, {
    "scenario": "LLM-guided",
    "mean_energy_feasible": float(U_llm[~infeasible_mask].mean()),
    "mean_energy_infeasible": float(U_llm[infeasible_mask].mean()),
    "energy_gap_infeasible_minus_feasible": float(U_llm[infeasible_mask].mean() - U_llm[~infeasible_mask].mean())
}]
energy_df = pd.DataFrame(energy_rows)
energy_df.to_csv(TABLE_DIR / "effective_energy_landscape_summary.csv", index=False)

display(energy_df)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[LLM MECHANISM: ENERGY LANDSCAPE RECONSTRUCTION]\n")
    f.write("Effective energy landscapes U=-log(rho) were reconstructed with and without the LLM operator.\n")
    f.write("The energy shift map tests whether the LLM behaves as a structural redistribution operator that changes basins of attraction.\n")
    f.write("Saved files: effective_energy_no_llm.npy, effective_energy_llm.npy, effective_energy_shift.npy, effective_energy_landscape_summary.csv.\n")
    f.write("Saved figure: figure_21_effective_energy_landscape_reconstruction.png.\n\n")

## Final Notebook-Level Synthesis
This final cell updates the outputs summary with the new scientific framing: controlled regimes, Pareto trade-offs, and LLM mechanism identification.

In [ ]:
# =========================================================
# FINAL SYNTHESIS FOR UPGRADED NOTEBOOK
# =========================================================
with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[UPGRADED FINAL SYNTHESIS]\n")
    f.write("The notebook was upgraded from a descriptive simulation into a controlled decision-dynamics experiment.\n")
    f.write("First, regime analysis maps how diffusion and drift parameters affect convergence, infeasibility, and oscillation.\n")
    f.write("Second, Pareto analysis treats tourism decision intelligence as a multi-objective trade-off among objective value, feasibility, entropy, and stability.\n")
    f.write("Third, LLM mechanism diagnostics test whether the LLM operator behaves as a projection, contraction, or structural redistribution mechanism.\n")
    f.write("These additions strengthen the notebook by replacing isolated metric observations with phase behavior, trade-off geometry, and mechanism-level interpretation.\n\n")

repro_extra = {
    "new_figures": [
        "figure_12_phase_map_optimality_gap.png",
        "figure_13_phase_map_infeasible_mass.png",
        "figure_14_phase_map_tail_tv.png",
        "figure_15_llm_strength_feasibility.png",
        "figure_16_llm_strength_entropy.png",
        "figure_17_pareto_feasibility_entropy.png",
        "figure_18_pareto_objective_stability.png",
        "figure_19_llm_persistence_infeasible_mass.png",
        "figure_20_llm_persistence_entropy.png",
        "figure_21_effective_energy_landscape_reconstruction.png"
    ],
    "new_tables": [
        "regime_phase_analysis.csv",
        "regime_phase_summary.csv",
        "llm_projection_strength_sweep.csv",
        "pareto_tradeoff_analysis.csv",
        "pareto_frontier_points.csv",
        "llm_marginal_tradeoff_rates.csv",
        "llm_persistence_after_removal.csv",
        "effective_energy_landscape_summary.csv"
    ]
}

with open(OUTPUT_DIR / "v5_added_artifacts.json", "w", encoding="utf-8") as f:
    json.dump(repro_extra, f, indent=2)

print("Notebook v5 upgrades completed.")
print("Additional artifacts summary saved to:", OUTPUT_DIR / "v5_added_artifacts.json")

## Quantitative Law Extraction: Entropy and Convergence Scaling
This section converts trajectory curves into fitted laws. Entropy, infeasible mass, and optimality gap are fitted to exponential relaxation forms so that the notebook reports decay rates rather than only visual trends.

In [ ]:
# =========================================================
# QUANTITATIVE LAW EXTRACTION: ENTROPY AND CONVERGENCE SCALING
# =========================================================
from pathlib import Path

# Reload dynamic metrics if the notebook state was restarted.
if "metrics_df" not in globals():
    dynamic_metrics_path = TABLE_DIR / "dynamic_metrics.csv"
    if dynamic_metrics_path.exists():
        metrics_df = pd.read_csv(dynamic_metrics_path)
    else:
        raise FileNotFoundError("dynamic_metrics.csv not found. Run the trajectory-metrics cells first.")


def fit_exponential_relaxation(group, metric_col, step_col="step", tail_fraction=0.20):
    """Fit y(t) ≈ y_inf + A exp(-k t) using a robust log-linear approximation."""
    data = group[[step_col, metric_col]].dropna().sort_values(step_col)
    t = data[step_col].to_numpy(dtype=float)
    y = data[metric_col].to_numpy(dtype=float)

    if len(t) < 8 or np.allclose(y, y[0]):
        return {
            "metric": metric_col,
            "decay_rate_k": np.nan,
            "asymptote_y_inf": float(y[-1]) if len(y) else np.nan,
            "amplitude_A": np.nan,
            "r2_log_fit": np.nan,
            "fit_status": "insufficient_variation"
        }

    tail_n = max(3, int(len(y) * tail_fraction))
    y_inf = float(np.median(y[-tail_n:]))

    # For decreasing curves, y - y_inf should be positive. Add a small adaptive offset.
    z = y - y_inf
    positive_scale = max(float(np.nanmax(np.abs(y))), 1.0)
    z = np.maximum(z, EPS * positive_scale)

    log_z = np.log(z)
    slope, intercept = np.polyfit(t, log_z, 1)
    pred_log = intercept + slope * t
    ss_res = float(np.sum((log_z - pred_log) ** 2))
    ss_tot = float(np.sum((log_z - np.mean(log_z)) ** 2))
    r2 = 1.0 - ss_res / (ss_tot + EPS)

    return {
        "metric": metric_col,
        "decay_rate_k": float(max(0.0, -slope)),
        "asymptote_y_inf": y_inf,
        "amplitude_A": float(np.exp(intercept)),
        "r2_log_fit": r2,
        "fit_status": "ok"
    }

scaling_rows = []
metrics_to_fit = [c for c in ["entropy", "infeasible_mass", "optimality_gap", "regret"] if c in metrics_df.columns]

for model_name, group in metrics_df.groupby("model"):
    if len(group) < 8:
        continue
    for metric_col in metrics_to_fit:
        row = fit_exponential_relaxation(group, metric_col)
        row["model"] = model_name
        row["n_points"] = int(len(group))
        scaling_rows.append(row)

scaling_laws_df = pd.DataFrame(scaling_rows)
scaling_laws_df = scaling_laws_df[["model", "metric", "decay_rate_k", "asymptote_y_inf", "amplitude_A", "r2_log_fit", "fit_status", "n_points"]]
scaling_laws_df.to_csv(TABLE_DIR / "scaling_laws_entropy_convergence.csv", index=False)

# Plot observed and fitted entropy curves for dynamic models.
plt.figure(figsize=(8.5, 5.5))
for model_name, group in metrics_df.groupby("model"):
    group = group.sort_values("step")
    if "entropy" not in group.columns:
        continue
    plt.plot(group["step"], group["entropy"], label=f"{model_name} observed")

    fit_row = scaling_laws_df[(scaling_laws_df["model"] == model_name) & (scaling_laws_df["metric"] == "entropy")]
    if len(fit_row) and fit_row.iloc[0]["fit_status"] == "ok":
        fr = fit_row.iloc[0]
        t = group["step"].to_numpy(dtype=float)
        y_fit = fr["asymptote_y_inf"] + fr["amplitude_A"] * np.exp(-fr["decay_rate_k"] * t)
        plt.plot(t, y_fit, linestyle="--", alpha=0.75, label=f"{model_name} fitted")

plt.xlabel("Decision step")
plt.ylabel("Entropy")
plt.title("Figure 22: Fitted entropy-relaxation laws")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_22_entropy_scaling_laws.png", dpi=300, bbox_inches="tight")
plt.show()

# Plot decay-rate comparison.
rate_plot_df = scaling_laws_df[scaling_laws_df["fit_status"] == "ok"].copy()
if not rate_plot_df.empty:
    plt.figure(figsize=(9, 5.5))
    for metric_col, g in rate_plot_df.groupby("metric"):
        plt.scatter(g["model"], g["decay_rate_k"], label=metric_col)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Fitted decay rate k")
    plt.title("Figure 23: Estimated convergence and entropy decay rates")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "figure_23_decay_rate_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()

try:
    from IPython.display import display
    display(scaling_laws_df)
except Exception:
    print(scaling_laws_df)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[SCALING LAWS: ENTROPY AND CONVERGENCE]\n")
    f.write("Trajectory curves were converted into fitted exponential-relaxation laws of the form y(t)=y_inf + A exp(-k t).\n")
    f.write("The fitted decay rate k provides a quantitative law for entropy reduction, infeasibility decay, regret reduction, and optimality-gap convergence.\n")
    f.write("Saved file: scaling_laws_entropy_convergence.csv.\n")
    f.write("Saved figures: figure_22_entropy_scaling_laws.png and figure_23_decay_rate_comparison.png.\n\n")


## Dimensionless Control Variables and Invariant Regime View
This section replaces raw parameters by normalized control ratios. The goal is to identify whether behavior is governed by invariant combinations such as drift-to-diffusion and LLM-strength-to-diffusion ratios.

In [ ]:
# =========================================================
# DIMENSIONLESS CONTROL VARIABLES AND INVARIANT REGIME VIEW
# =========================================================
if "regime_df" not in globals():
    regime_path = TABLE_DIR / "regime_phase_analysis.csv"
    if regime_path.exists():
        regime_df = pd.read_csv(regime_path)
    else:
        raise FileNotFoundError("regime_phase_analysis.csv not found. Run the regime-analysis cell first.")

invariant_df = regime_df.copy()
invariant_df["kappa_drift_diffusion"] = invariant_df["drift_scale"] / (invariant_df["diffusion"] + EPS)
invariant_df["kappa_llm_diffusion"] = invariant_df["lambda_shape"] / (invariant_df["diffusion"] + EPS)
invariant_df["projection_to_diffusion"] = invariant_df["projection_strength"] / (invariant_df["diffusion"] + EPS)
invariant_df["normalized_entropy"] = invariant_df["final_entropy"] / np.log(P_static.size + EPS) if "P_static" in globals() else invariant_df["final_entropy"] / np.log(N * N)
invariant_df["normalized_gap"] = invariant_df["final_optimality_gap"] / (abs(float(J.max() - J.min())) + EPS)
invariant_df.to_csv(TABLE_DIR / "dimensionless_invariant_regime_variables.csv", index=False)

plt.figure(figsize=(8, 5.8))
scatter = plt.scatter(
    np.log10(invariant_df["kappa_drift_diffusion"] + EPS),
    invariant_df["normalized_entropy"],
    c=invariant_df["final_infeasible_mass"],
    s=70,
    alpha=0.85
)
plt.colorbar(scatter, label="Final infeasible mass")
plt.xlabel("log10(||v|| / D)")
plt.ylabel("Normalized final entropy")
plt.title("Figure 24: Invariant view using drift-to-diffusion ratio")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_24_invariant_drift_diffusion_view.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5.8))
scatter = plt.scatter(
    np.log10(invariant_df["kappa_llm_diffusion"] + EPS),
    invariant_df["final_infeasible_mass"],
    c=invariant_df["mean_tail_tv"],
    s=70,
    alpha=0.85
)
plt.colorbar(scatter, label="Tail trajectory variation")
plt.xlabel("log10(lambda / D)")
plt.ylabel("Final infeasible mass")
plt.title("Figure 25: Invariant view using LLM-strength-to-diffusion ratio")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_25_invariant_llm_diffusion_view.png", dpi=300, bbox_inches="tight")
plt.show()

# Simple regime summary by quantiles of kappa_drift_diffusion.
invariant_df["kappa_drift_bin"] = pd.qcut(invariant_df["kappa_drift_diffusion"], q=4, duplicates="drop")
invariant_summary = invariant_df.groupby("kappa_drift_bin", observed=False).agg(
    mean_final_entropy=("final_entropy", "mean"),
    mean_normalized_entropy=("normalized_entropy", "mean"),
    mean_infeasible_mass=("final_infeasible_mass", "mean"),
    mean_tail_tv=("mean_tail_tv", "mean"),
    mean_optimality_gap=("final_optimality_gap", "mean"),
    n=("final_entropy", "size")
).reset_index()
invariant_summary.to_csv(TABLE_DIR / "dimensionless_invariant_summary.csv", index=False)

try:
    from IPython.display import display
    display(invariant_summary)
except Exception:
    print(invariant_summary)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[DIMENSIONLESS INVARIANT CONTROL VARIABLES]\n")
    f.write("Raw parameters were converted into invariant ratios such as ||v||/D and lambda/D to test whether behavior collapses into scale-independent regimes.\n")
    f.write("Saved files: dimensionless_invariant_regime_variables.csv and dimensionless_invariant_summary.csv.\n")
    f.write("Saved figures: figure_24_invariant_drift_diffusion_view.png and figure_25_invariant_llm_diffusion_view.png.\n\n")


## Entropy-Based Causal Decomposition of the LLM Operator
This section isolates the causal role of each operator component. The dynamics are run with base drift--diffusion only, projection only, shaping only, and the full LLM operator to quantify which component drives entropy reduction, feasibility improvement, and objective change.

In [ ]:
# =========================================================
# ENTROPY-BASED CAUSAL DECOMPOSITION OF THE LLM OPERATOR
# =========================================================
DECOMP_STEPS = 120


def step_with_operator_component(P, component):
    """Apply one base dynamics step followed by a selected operator component."""
    Q = step_variational(P, use_llm=False)
    if component == "base_only":
        return normalize(Q)
    if component == "projection_only":
        return projection_operator(Q, projection_strength=0.85)
    if component == "shaping_only":
        return soft_shaping_operator(Q, lambda_shape=1.7, penalty_boost=2.0)
    if component == "full_operator":
        return llm_policy_operator(Q, lambda_shape=1.7, projection_strength=0.85, penalty_boost=2.0)
    raise ValueError(f"Unknown component: {component}")

component_labels = {
    "base_only": "Base dynamics only",
    "projection_only": "Projection only",
    "shaping_only": "Shaping only",
    "full_operator": "Full LLM operator"
}

decomp_rows = []
for component in component_labels:
    Pdist = pi0.copy()
    previous = None
    for t in range(DECOMP_STEPS + 1):
        if t > 0:
            previous = Pdist.copy()
            Pdist = step_with_operator_component(Pdist, component)
        decomp_rows.append({
            "step": t,
            "component": component,
            "component_label": component_labels[component],
            "entropy": entropy(Pdist),
            "infeasible_mass": infeasible_mass(Pdist),
            "expected_J": expected_J(Pdist),
            "optimality_gap": optimality_gap(Pdist),
            "regret": regret(Pdist),
            "stability_tv": total_variation(Pdist, previous) if previous is not None else 0.0,
            "max_probability": float(Pdist.max())
        })

causal_df = pd.DataFrame(decomp_rows)
causal_df.to_csv(TABLE_DIR / "causal_operator_decomposition_timeseries.csv", index=False)

initial_row = causal_df[causal_df["step"] == 0].iloc[0]
final_rows = causal_df[causal_df["step"] == DECOMP_STEPS].copy()
causal_summary_rows = []
for _, row in final_rows.iterrows():
    causal_summary_rows.append({
        "component": row["component"],
        "component_label": row["component_label"],
        "final_entropy": row["entropy"],
        "entropy_reduction_from_initial": initial_row["entropy"] - row["entropy"],
        "final_infeasible_mass": row["infeasible_mass"],
        "infeasible_reduction_from_initial": initial_row["infeasible_mass"] - row["infeasible_mass"],
        "final_expected_J": row["expected_J"],
        "objective_reduction_from_initial": initial_row["expected_J"] - row["expected_J"],
        "final_regret": row["regret"],
        "mean_stability_tv": causal_df[causal_df["component"] == row["component"]]["stability_tv"].mean(),
        "max_probability": row["max_probability"]
    })

causal_summary_df = pd.DataFrame(causal_summary_rows)

# Attribute incremental effects relative to base dynamics.
base_final = causal_summary_df[causal_summary_df["component"] == "base_only"].iloc[0]
causal_summary_df["entropy_reduction_vs_base"] = base_final["final_entropy"] - causal_summary_df["final_entropy"]
causal_summary_df["infeasible_reduction_vs_base"] = base_final["final_infeasible_mass"] - causal_summary_df["final_infeasible_mass"]
causal_summary_df["objective_reduction_vs_base"] = base_final["final_expected_J"] - causal_summary_df["final_expected_J"]
causal_summary_df.to_csv(TABLE_DIR / "causal_operator_decomposition_summary.csv", index=False)

plt.figure(figsize=(8.5, 5.5))
for label, group in causal_df.groupby("component_label"):
    plt.plot(group["step"], group["entropy"], label=label)
plt.xlabel("Decision step")
plt.ylabel("Entropy")
plt.title("Figure 26: Causal decomposition of entropy dynamics")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_26_causal_entropy_decomposition.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8.5, 5.5))
for label, group in causal_df.groupby("component_label"):
    plt.plot(group["step"], group["infeasible_mass"], label=label)
plt.xlabel("Decision step")
plt.ylabel("Infeasible probability mass")
plt.title("Figure 27: Causal decomposition of feasibility correction")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_27_causal_feasibility_decomposition.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5.5))
plot_df = causal_summary_df.sort_values("infeasible_reduction_vs_base", ascending=False)
plt.bar(plot_df["component_label"], plot_df["infeasible_reduction_vs_base"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Final infeasible-mass reduction vs. base")
plt.title("Figure 28: Component-level causal contribution to feasibility")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_28_component_feasibility_contribution.png", dpi=300, bbox_inches="tight")
plt.show()

try:
    from IPython.display import display
    display(causal_summary_df)
except Exception:
    print(causal_summary_df)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[CAUSAL OPERATOR DECOMPOSITION]\n")
    f.write("The LLM operator was decomposed into base dynamics, projection only, shaping only, and full operator variants.\n")
    f.write("Each component was evaluated for entropy reduction, infeasible-mass reduction, objective reduction, regret, and trajectory stability.\n")
    f.write("This allows causal attribution of the LLM effect instead of only reporting aggregate improvement.\n")
    f.write("Saved files: causal_operator_decomposition_timeseries.csv and causal_operator_decomposition_summary.csv.\n")
    f.write("Saved figures: figure_26_causal_entropy_decomposition.png, figure_27_causal_feasibility_decomposition.png, and figure_28_component_feasibility_contribution.png.\n\n")


## Final Law-Oriented Synthesis
This final cell records the additional v6 artifacts and explicitly states how the notebook has moved from descriptive outputs to laws, invariants, and causal attribution.

In [ ]:
# =========================================================
# FINAL LAW-ORIENTED SYNTHESIS AND ARTIFACT INDEX
# =========================================================
v6_artifacts = {
    "new_figures": [
        "figure_22_entropy_scaling_laws.png",
        "figure_23_decay_rate_comparison.png",
        "figure_24_invariant_drift_diffusion_view.png",
        "figure_25_invariant_llm_diffusion_view.png",
        "figure_26_causal_entropy_decomposition.png",
        "figure_27_causal_feasibility_decomposition.png",
        "figure_28_component_feasibility_contribution.png"
    ],
    "new_tables": [
        "scaling_laws_entropy_convergence.csv",
        "dimensionless_invariant_regime_variables.csv",
        "dimensionless_invariant_summary.csv",
        "causal_operator_decomposition_timeseries.csv",
        "causal_operator_decomposition_summary.csv"
    ]
}

with open(OUTPUT_DIR / "v6_added_artifacts.json", "w", encoding="utf-8") as f:
    json.dump(v6_artifacts, f, indent=2)

with open(OUTPUT_SUMMARY_PATH, "a", encoding="utf-8") as f:
    f.write("[V6 FINAL SYNTHESIS: LAWS, INVARIANTS, AND CAUSAL ATTRIBUTION]\n")
    f.write("The notebook was further upgraded from phase and Pareto diagnostics to law-oriented analysis.\n")
    f.write("First, entropy, infeasibility, regret, and optimality-gap trajectories were fitted to exponential relaxation laws to estimate decay rates.\n")
    f.write("Second, raw parameters were transformed into dimensionless ratios such as ||v||/D and lambda/D to support scale-independent interpretation.\n")
    f.write("Third, the LLM operator was decomposed into projection-only, shaping-only, and full-operator variants to attribute causal effects on entropy, feasibility, and objective improvement.\n")
    f.write("These additions strengthen the results by replacing descriptive curves with fitted laws, invariant control variables, and mechanism-level causal decomposition.\n\n")

print("Notebook v6 upgrades completed.")
print("Additional artifacts summary saved to:", OUTPUT_DIR / "v6_added_artifacts.json")
